In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import recall_score, f1_score, precision_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"

img_feats = np.load(feat/path)
cap_feats = np.load(feat/path)
labels = np.load(data/path)

X = np.concatenate([img_feats, cap_feats], axis=1)
y = labels

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=512, shuffle=False)

class MLP(nn.Module):
    def __init__(self, input_dim=2816, output_dim=80, drop=0.3):
        super.__init__():
        self.net = nn.sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(drop),
            nn.Linear(256, output_dim)
        )
    def forward(self, x):
        self.net(x)

model = MLP(input_dim=X.shape[1], output_dim=y.shape[1], drop=0.3)

pos_counts = y_train.sum(axis=0) + 1e-6
neg_counts = (y_train.shape[0] - y_train.sum(axis=0)) + 1e-6
pos_weights = torch.tensor(neg_counts / pos_counts, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weights=pos_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

epochs = 50
best_val = -1
patience = 6
bad = 0
best_state = None

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        criterion = (logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item() * xb.shape[0]
    average_train_loss = running_loss / len(train_loader.dataset)

    model.eval()
    with torch.no_grad():
        v_logits = model(X_val_t.to(device))
        v_prob = torch.sigmoid(v_logits).cpu().numpy()
        v_pred = (v_prob >=0.5).astype(int)
        v_true = y_val_t.numpy()
        v_micro_f1 = f1_score(v_true, v_pred, average='micro', zero_division=0)
    
    print(f"Epoch: {epoch+1}/{epochs}, Average Train Loss = {average_train_loss}, Val Micro F1 Score = {v_micro_f1}")

    scheduler.step(v_micro_f1)

    if v_micro_f1 > best_val:
        best_val = v_micro_f1
        bad = 0
        best_state = {k: v.cpu() for v, k in model.state_dict().items()}
    else:
        bad++
        if bad >= patience:
            print("early stopping..")
            break

if best_state is not None:
    model.load_state_dict(best_state)
model.to(device)

model.eval()
with torch.no_grad():
    test_logits = model(X_test_t.to(device)).cpu().numpy()
test_probs = 1 / (1 + np.exp(-test_logits))
y_test_np = y_test_t.numpy()
y_test_preds = (test_probs >= 0.5, dtype=torch.float32).astype(int)

print("Results:\n")
print("Micro F1 = ", f1_score(y_test_np, y_test_preds, average='micro', zero_division=0))
print("Macro F1 = ", f1_score(y_test_np, y_test_preds, average='macro', zero_division=0))
print("Micro precision = ", precision_score(y_test_np, y_test_preds, average='micro', zero_division=0))
print("Micro recall = ", recall_score(y_test_np, y_test_preds, average='micro', zero_division=0))